Implementing the Macro method from Couloumbe (2024)

Data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import TimeSeriesSplit
import statsmodels.api as sm

In [ ]:
df = pd.read_csv('england_master.csv')

In [ ]:
df['date'] = pd.to_datetime(df['Unnamed: 0'].str.replace('Q', '-Q'))
df.set_index('date', inplace=True)

In [ ]:
df_clean = df[['starts', 'hprice', 'cc', 'rate', 'vol', 'gdp_def']].copy()

In [ ]:
df_clean['starts_lag1'] = df_clean['starts'].shift(1)
df_clean['starts_lag4'] = df_clean['starts'].shift(4)
df_clean['vol_lag1'] = df_clean['vol'].shift(1)
df_clean['rate_lag1'] = df_clean['rate'].shift(1)
df_clean['time_trend'] = np.arange(len(df_clean))

In [ ]:
df_clean = df_clean.dropna()

In [ ]:
df_clean['starts'] = np.log(df_clean['starts'])
df_clean['hprice'] = np.log(df_clean['hprice'] / df_clean['gdp_def'])  # deflated by gdp_def, matching 04_ARRF.py
df_clean['cc'] = np.log(df_clean['cc'] / df_clean['gdp_def'])  # deflated by gdp_def, matching 04_ARRF.py

In [ ]:
df_clean['margin'] = df_clean['hprice'] - df_clean['cc']  # log(hprice) - log(cc) = log(price/cost); combines the two collinear regressors flagged in the diagnostic battery below into a single price-cost spread

In [ ]:
y = df_clean['starts']  # log(starts) now that the log transform runs first, consistent with lhstarts convention

In [ ]:
# OPEN DESIGN QUESTION: S/X split is reversed vs Coulombe's AR special case (AR lags in forest state S here, margin/rate in linear block X) -- deliberate MRF-style variant, left as-is, not a strict ARRF replication
X = df_clean[['margin', 'rate']]
X = sm.add_constant(X)

In [ ]:
S = df_clean[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]
print(f"Data cleaned. Proceeding with {len(y)} perfectly aligned quarters.")

The macroeconomic RF

In [ ]:
rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf.fit(S, y)

In [ ]:
leaf_assignments = rf.apply(S)

In [ ]:
valid_idx = df_clean.index
gtvps = pd.DataFrame(index=valid_idx, columns=X.columns, dtype=float)
predicted_y = pd.Series(index=valid_idx, dtype=float)

In [ ]:
# Freeze ridge lambda via leaf-weighted CV on the pre-2010Q1 subsample only: a flat unweighted TimeSeriesSplit fit picked lambda at the edge of the grid and zeroed every slope, so each candidate alpha is scored using the same leaf-co-occurrence weights as the main loop (restricted to each fold's training positions), closer to Friedberg et al.'s local-linear-forest tuning than a single global fit
X_no_const = X.drop(columns='const')
pre_idx = np.where(df_clean.index < pd.Timestamp('2010-01-01'))[0]
alphas = np.logspace(-3, 5, 17)
tscv = TimeSeriesSplit(n_splits=5)
cv_errors = {a: [] for a in alphas}
for train_pos, test_pos in tscv.split(pre_idx):
    train_idx = pre_idx[train_pos]
    for t in pre_idx[test_pos]:
        w = np.sum(leaf_assignments[train_idx] == leaf_assignments[t], axis=1)
        if w.sum() == 0:
            continue
        w = w / w.sum()
        for a in alphas:
            m = Ridge(alpha=a).fit(X_no_const.iloc[train_idx], y.iloc[train_idx], sample_weight=w)
            pred = m.predict(X_no_const.iloc[[t]])[0]
            cv_errors[a].append((y.iloc[t] - pred) ** 2)
lam = min(alphas, key=lambda a: np.mean(cv_errors[a]))
print(f"Selected ridge lambda (leaf-weighted CV, pre-2010Q1): {lam}")

In [ ]:
# Leaf-local ridge WLS (Friedberg et al. 2020 local linear forest), replacing the unregularized sm.WLS flagged above
for t_idx in range(len(valid_idx)):
  current_leaves = leaf_assignments[t_idx, :]
  weights = np.sum(leaf_assignments == current_leaves, axis=1)
  weights = weights / weights.sum()
  ridge_model = Ridge(alpha=lam).fit(X_no_const, y, sample_weight=weights)
  gtvps.iloc[t_idx] = np.concatenate([[ridge_model.intercept_], ridge_model.coef_])
  predicted_y.iloc[t_idx] = ridge_model.predict(X_no_const.iloc[[t_idx]])[0]

In [ ]:
gtvps = gtvps.astype(float)

Visualisation


In [ ]:
#Plot1
plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['rate'], color='crimson', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of the BoE Base Rate on Housing Starts', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Rate)', fontsize=12)
plt.grid(True, alpha=0.3)

plt.fill_between(gtvps.index, gtvps['rate'], 0, where=(gtvps['rate'] < 0), color='crimson', alpha=0.1)

plt.tight_layout()
plt.savefig('gtvp_interest_rate.png')
plt.show()

In [ ]:
#Plot2
plt.figure(figsize=(12, 6))
plt.plot(gtvps.index, gtvps['margin'], color='darkgreen', linewidth=2.5)
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.title('GTVP: The Time-Varying Impact of the Price-Cost Margin', fontsize=14, fontweight='bold')
plt.ylabel('Coefficient (Elasticity of Margin)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gtvp_margin.png')
plt.show()

## Diagnostic battery

Full GTVP diagnostic battery — tests whether the fitted specification's local design matrix is well-conditioned, whether local sample sizes are sufficient, and whether the model's out-of-sample predictive accuracy significantly improves on an intercept-only baseline.

In [ ]:
from scipy import stats

### 1. Collinearity diagnostics

In [ ]:
corr_margin_rate = df_clean[['margin', 'rate']].corr().loc['margin', 'rate']
print(f"Global correlation between margin and rate (log margin vs level rate): {corr_margin_rate:.4f}")

In [ ]:
# Local design-matrix condition number per terminal leaf/node of the fitted forest (reuses rf/leaf_assignments, no refit)
n_trees = leaf_assignments.shape[1]
leaf_conds = []
for t in range(n_trees):
    tree_leaves = leaf_assignments[:, t]
    for leaf_id in np.unique(tree_leaves):
        idx = np.where(tree_leaves == leaf_id)[0]
        if len(idx) < X.shape[1]:
            continue
        Xl = X.iloc[idx].values
        singular_values = np.linalg.svd(Xl, compute_uv=False)
        if singular_values.min() > 0:
            leaf_conds.append(singular_values.max() / singular_values.min())
leaf_conds = np.array(leaf_conds)
pct_above_100 = 100 * np.mean(leaf_conds > 100)
print(f"Leaves evaluated (>= {X.shape[1]} obs, across {n_trees} trees): {len(leaf_conds)}")
print(f"Local design-matrix condition number - min: {leaf_conds.min():.1f}, median: {np.median(leaf_conds):.1f}, max: {leaf_conds.max():.1f}")
print(f"% of leaves with condition number > 100: {pct_above_100:.1f}%")

In [ ]:
global_singular_values = np.linalg.svd(X.values, compute_uv=False)
global_cond = global_singular_values.max() / global_singular_values.min()
print(f"Global condition number of design matrix X (const, margin, rate): {global_cond:.1f}")

### 2. Sample-size sufficiency check

In [ ]:
# Effective sample size (ESS) of the leaf-co-occurrence weights actually used in the local ridge fit (cell 17), vs the 3 parameters being estimated
ess_per_point = np.empty(len(valid_idx))
for t_idx in range(len(valid_idx)):
    current_leaves = leaf_assignments[t_idx, :]
    weights = np.sum(leaf_assignments == current_leaves, axis=1).astype(float)
    weights = weights / weights.sum()
    ess_per_point[t_idx] = 1.0 / np.sum(weights ** 2)

n_params = 3  # const, margin, rate
pct_below_params = 100 * np.mean(ess_per_point < n_params)
pct_below_4x = 100 * np.mean(ess_per_point < 4 * n_params)
print(f"Effective sample size (ESS) per local regression - min: {ess_per_point.min():.1f}, median: {np.median(ess_per_point):.1f}, max: {ess_per_point.max():.1f}")
print(f"Number of parameters being estimated per local fit: {n_params}")
print(f"% of local regressions with ESS < {n_params} (params): {pct_below_params:.1f}%")
print(f"% of local regressions with ESS < {4 * n_params} (4x params): {pct_below_4x:.1f}%")
if ess_per_point.min() >= n_params:
    print("Every local regression has ESS >= number of parameters: a small-sample bottleneck is ruled out as a standalone explanation for near-zero coefficients, if any.")
else:
    print("Some local regressions have ESS below the number of parameters: sample size cannot be ruled out as a contributing factor.")

### 3. Predictive performance comparison

In [ ]:
# Out-of-sample CV: ridge-penalized GTVP (leaf-weighted local ridge, alpha=lam, reusing the fitted rf/leaf_assignments)
# vs a nested intercept-only baseline that uses the SAME leaf weights as GTVP but drops margin/rate.
# This is the correct nested restriction for a Clark-West test: a flat, unweighted global mean would use a
# different information set (no leaf localization) and conflate "GTVP beats a naive predictor" with
# "GTVP's margin/rate elasticities add value", which is the actual question this test is meant to answer.
tscv_eval = TimeSeriesSplit(n_splits=5)
n_obs = len(y)
cv_records = []
for fold_id, (train_pos, test_pos) in enumerate(tscv_eval.split(np.arange(n_obs))):
    y_train = y.iloc[train_pos]
    for t in test_pos:
        w = np.sum(leaf_assignments[train_pos] == leaf_assignments[t], axis=1).astype(float)
        if w.sum() == 0:
            w = np.ones(len(train_pos))
        w = w / w.sum()
        restricted_pred = np.average(y_train, weights=w)
        m = Ridge(alpha=lam).fit(X_no_const.iloc[train_pos], y_train, sample_weight=w)
        gtvp_pred = m.predict(X_no_const.iloc[[t]])[0]
        cv_records.append({
            'fold': fold_id,
            'y_true': y.iloc[t],
            'gtvp_pred': gtvp_pred,
            'restricted_pred': restricted_pred,
        })
cv_df = pd.DataFrame(cv_records)
cv_df['gtvp_sq_err'] = (cv_df['y_true'] - cv_df['gtvp_pred']) ** 2
cv_df['restricted_sq_err'] = (cv_df['y_true'] - cv_df['restricted_pred']) ** 2
print(f"Out-of-sample folds: {cv_df['fold'].nunique()}, total held-out observations: {len(cv_df)}")

In [ ]:
gtvp_cv_mse = cv_df['gtvp_sq_err'].mean()
restricted_cv_mse = cv_df['restricted_sq_err'].mean()
print(f"CV MSE - ridge-penalized GTVP (alpha={lam}): {gtvp_cv_mse:.6f}")
print(f"CV MSE - leaf-weighted intercept-only baseline (nested, same weights, no margin/rate): {restricted_cv_mse:.6f}")
print(f"Difference (GTVP - intercept-only): {gtvp_cv_mse - restricted_cv_mse:.6f}")

t_stat, p_val_ttest = stats.ttest_rel(cv_df['gtvp_sq_err'], cv_df['restricted_sq_err'])
print(f"Paired t-test on per-observation squared-error differences: t={t_stat:.4f}, two-sided p={p_val_ttest:.4f}")
if p_val_ttest < 0.05:
    print("GTVP and the intercept-only baseline are statistically distinguishable at the 5% level (paired t-test).")
else:
    print("GTVP and the intercept-only baseline are NOT statistically distinguishable at the 5% level (paired t-test); see the Clark-West test below for the formal nested-model comparison.")

### 4. Formal significance test

We use the Clark-West (2007) test as the primary formal significance test comparing the GTVP model against the intercept-only restricted model. Clark-West corrects for the fact that, under the null of no genuine time-varying effect, a nested unrestricted model's out-of-sample MSE is biased to look better purely from having more free parameters — a plain MSE-difference test is not valid for nested model comparisons.

The restricted model uses the *same* leaf-co-occurrence weights as GTVP (so it shares GTVP's autoregressive/localization information from the forest state `S`) but drops margin and rate, leaving only a weighted intercept. This is the correct nested restriction: a flat, unweighted global-mean baseline would use a different information set entirely, and any GTVP win against it would conflate genuine margin/rate elasticities with the RF's leaf-based localization — exactly the confound this diagnostic battery is trying to rule out.

We deliberately do **not** use a fold-level Wilcoxon signed-rank test (only n=5 fold-level MSE observations here — badly underpowered) or a per-observation Wilcoxon test (invalid in this setting: paired squared-error differences are continuous floating-point values that are essentially never exactly tied, so any apparent "ties" the test would key off of are floating-point noise, not genuine ties). Neither test is used as a primary result anywhere in this notebook; this note documents why they are excluded from the citable results.

In [ ]:
cw_terms = cv_df['restricted_sq_err'] - cv_df['gtvp_sq_err'] + (cv_df['restricted_pred'] - cv_df['gtvp_pred']) ** 2
n_cw = len(cw_terms)
cw_mean = cw_terms.mean()
cw_se = cw_terms.std(ddof=1) / np.sqrt(n_cw)
cw_stat = cw_mean / cw_se
p_value_cw = 1 - stats.norm.cdf(cw_stat)
print("Clark-West test (GTVP vs intercept-only restricted model):")
print(f"  n = {n_cw}")
print(f"  CW statistic = {cw_stat:.4f}")
print(f"  one-sided p-value (H1: GTVP improves on restricted model) = {p_value_cw:.4f}")
if p_value_cw < 0.05:
    print("Reject H0 at the 5% level: GTVP shows a statistically significant improvement over the intercept-only model.")
else:
    print("Fail to reject H0 at the 5% level: no statistically significant evidence that GTVP improves on the intercept-only model.")

### 5. Coefficient / plotting sanity check

In [ ]:
coef_summary = gtvps[['margin', 'rate']].abs().agg(['median', 'mean', 'std', 'max'])
print("Magnitude of GTVP slope coefficients (absolute value, across all quarters):")
print(coef_summary)
print()
print(f"Selected ridge penalty lambda = {lam:.1f}, at the top edge of the search grid ({alphas.min():.1e} to {alphas.max():.1e}).")
print(f"Global correlation(margin, rate) = {corr_margin_rate:.4f}; median local design-matrix condition number = {np.median(leaf_conds):.1f} ({pct_above_100:.1f}% of leaves > 100).")
if pct_above_100 > 50 or abs(corr_margin_rate) > 0.8:
    print("Coefficient magnitudes above remain shrunk by local ill-conditioning and/or margin-rate collinearity combined with the heavily-regularized ridge fit; collapsing hprice/cc into margin has not, on its own, resolved the shrinkage problem diagnosed for the original specification.")
else:
    print("Collapsing hprice and cc into margin has resolved most of the local ill-conditioning that shrank the previous specification's coefficients toward zero; whether the remaining coefficient magnitudes are practically meaningful should be judged against the CV/Clark-West results above, not this diagnostic alone.")

### 6. Residual diagnostics

In [ ]:
# In-sample residuals from the already-fitted local ridge model (predicted_y from cell 17, no refit)
residuals = y - predicted_y

gfc_mask = (df_clean.index >= '2007-07-01') & (df_clean.index <= '2009-06-30')
calm_mask = (df_clean.index >= '2013-01-01') & (df_clean.index <= '2014-12-31')

gfc_resid = residuals[gfc_mask]
calm_resid = residuals[calm_mask]
print(f"GFC period residuals (2007Q3-2009Q2): n={len(gfc_resid)}")
print(f"Calmer benchmark residuals (2013Q1-2014Q4): n={len(calm_resid)}")

levene_stat, levene_p = stats.levene(gfc_resid, calm_resid)
print(f"Levene's test for equal variance (GFC vs calmer benchmark): statistic={levene_stat:.4f}, p={levene_p:.4f}")
if levene_p < 0.05:
    print("Reject equal-variance null at the 5% level: residual variance differs significantly between the GFC period and the calmer benchmark (heteroskedasticity present).")
else:
    print("Fail to reject equal-variance null at the 5% level: no significant evidence of heteroskedasticity between the GFC period and the calmer benchmark.")

In [ ]:
# Per-fold CV MSE breakdown (reuses cv_df from the predictive-performance-comparison CV, no refit)
fold_mse = cv_df.groupby('fold').agg(
    n_obs=('gtvp_sq_err', 'size'),
    gtvp_mse=('gtvp_sq_err', 'mean'),
    restricted_mse=('restricted_sq_err', 'mean'),
)
fold_mse['share_of_total_gtvp_sq_err'] = (
    cv_df.groupby('fold')['gtvp_sq_err'].sum() / cv_df['gtvp_sq_err'].sum()
)
print("Per-fold CV MSE breakdown:")
print(fold_mse)

n_folds = fold_mse.shape[0]
max_share = fold_mse['share_of_total_gtvp_sq_err'].max()
dominant_fold = fold_mse['share_of_total_gtvp_sq_err'].idxmax()
equal_share = 1.0 / n_folds
print()
print(f"Largest single-fold share of total squared error: fold {dominant_fold} at {max_share:.1%} (equal split across {n_folds} folds would be {equal_share:.1%}).")
if max_share > 2 * equal_share:
    print(f"Flagged: fold {dominant_fold} contributes more than double an equal share of total CV squared error — GTVP CV performance is not evenly distributed across folds.")
else:
    print("No single fold dominates total CV squared error by more than 2x an equal share.")

### 7. Root-cause check — AR-dominance vs. collinearity

Switching from hprice/cc to margin roughly halved both the local and global design-matrix condition numbers (Section 1) but left the ridge penalty pinned at the top of the search grid, coefficient magnitudes at the same order of size, and the Clark-West result unchanged (Sections 3-5). That pattern — regularization maxed out and X's contribution flat regardless of which two regressors are in X — could mean the forest's autoregressive state `S` (lagged starts, vol, rate, time trend) already explains nearly all of the predictable variation in `y`, leaving little residual variance for the linear block to explain, independent of collinearity within X. This section tests that directly.

In [ ]:
# Persistence of the outcome itself: how much of log(starts) is just its own recent past?
ar1_corr = y.corr(y.shift(1))
ar4_corr = y.corr(y.shift(4))
print(f"Autocorrelation of log(starts): lag 1 = {ar1_corr:.4f}, lag 4 = {ar4_corr:.4f}")

In [ ]:
# What is the forest's leaf structure actually splitting on? (reuses fitted rf, no refit)
importances = pd.Series(rf.feature_importances_, index=S.columns).sort_values(ascending=False)
print("RF feature importances for the forest state S (what the leaves are actually built from):")
print(importances)

In [ ]:
# Out-of-sample R^2 decomposition (reuses cv_df from Section 3, no refit): how much of the achievable
# out-of-sample R^2 does the leaf-weighted mean of S alone already capture, before margin/rate are added?
ss_total = ((cv_df['y_true'] - cv_df['y_true'].mean()) ** 2).sum()
r2_restricted = 1 - cv_df['restricted_sq_err'].sum() / ss_total
r2_gtvp = 1 - cv_df['gtvp_sq_err'].sum() / ss_total
print(f"Out-of-sample R^2 - leaf-weighted mean using S alone (no margin/rate): {r2_restricted:.4f}")
print(f"Out-of-sample R^2 - ridge-penalized GTVP (S via leaves + margin/rate): {r2_gtvp:.4f}")
print(f"Incremental R^2 from adding margin/rate on top of S: {r2_gtvp - r2_restricted:.5f}")
if abs(r2_gtvp - r2_restricted) < 0.005:
    print(f"Adding margin/rate contributes essentially zero incremental out-of-sample R^2 on top of S ({r2_restricted:.1%} of variance explained by S alone, unchanged by X) -- not because S already saturates the achievable R^2 ({1 - r2_restricted:.1%} of variance remains unexplained), but because whatever residual variance is left has no detectable linear relationship with margin/rate in this specification. Ridge CV pinning lambda at the grid edge is therefore CV correctly detecting that X has no marginal predictive value here, not an overfitting artifact.")
else:
    print("Margin/rate contribute non-trivial incremental out-of-sample R^2; the near-zero coefficient magnitudes are not explained by a lack of genuine signal in X and warrant further investigation (e.g. numerical issues in the ridge fit).")

## Section 8: Respecification — does price/cost/rate history in S, or lagged growth rates in X, restore signal?

Section 7 showed the forest's regime state `S` is built almost entirely from housing-starts momentum (93.6% combined importance of `starts_lag1`/`starts_lag4`/`vol_lag1`) and that neither the hprice/cc pair nor margin/rate carries any detectable incremental predictive power once conditioned on those leaves. Two specification choices could plausibly explain a genuine null rather than a real absence of effect: (a) `S` never sees price/cost/rate history, so leaves cannot represent a price/cost/rate regime even if one exists; (b) `X` uses contemporaneous levels, when a construction start plausibly responds to price/cost/rate *changes* with a decision/permitting lag, not to this quarter's level. This section tests each change in isolation against the same nested-CV/Clark-West battery used in Sections 3-4, holding everything else fixed, and reports the null if it remains null.

In [ ]:
# Extra lagged/differenced price-cost-rate history, added on top of the already-cleaned df_clean.
# hprice_lag1/cc_lag1 mirror the existing single-period lag convention already used for rate_lag1/vol_lag1 in S.
# rate_chg_lag1/margin_growth_lag1 are differenced-then-lagged so they use only information available
# before quarter t's start decision (no contemporaneous leakage): diff(1) computes t-1 -> t growth,
# shift(1) moves that back one more period so it reflects t-2 -> t-1 growth, known at the start of t.
df8 = df_clean.copy()
df8['hprice_lag1'] = df8['hprice'].shift(1)
df8['cc_lag1'] = df8['cc'].shift(1)
df8['rate_chg_lag1'] = df8['rate'].diff(1).shift(1)
df8['margin_growth_lag1'] = df8['margin'].diff(1).shift(1)
df8 = df8.dropna()

y8 = df8['starts']
S_base8 = df8[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend']]
S_aug = df8[['starts_lag1', 'starts_lag4', 'vol_lag1', 'rate_lag1', 'time_trend', 'hprice_lag1', 'cc_lag1', 'rate_chg_lag1']]
X_level8 = sm.add_constant(df8[['margin', 'rate']])
X_growth8 = sm.add_constant(df8[['margin_growth_lag1', 'rate_chg_lag1']])
print(f"Section 8 sample: {len(y8)} quarters (vs {len(y)} in the main specification; {len(y) - len(y8)} lost to the extra lag needed for the differenced features).")

### 8a. Augment S with price/cost/rate history, keep X at contemporaneous levels

In [ ]:
rf_a = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf_a.fit(S_aug, y8)
leaf_assignments_a = rf_a.apply(S_aug)
importances_a = pd.Series(rf_a.feature_importances_, index=S_aug.columns).sort_values(ascending=False)
print("RF feature importances with hprice_lag1/cc_lag1/rate_chg_lag1 added to S:")
print(importances_a)

In [ ]:
# Freeze ridge lambda for this variant the same way as the main specification (cell 17): leaf-weighted CV
# on the pre-2010Q1 subsample, same alpha grid, reusing this variant's leaf co-occurrence weights.
X_level8_no_const = X_level8.drop(columns='const')
pre_idx8 = np.where(df8.index < pd.Timestamp('2010-01-01'))[0]
cv_errors_a = {a: [] for a in alphas}
for train_pos, test_pos in tscv.split(pre_idx8):
    train_idx = pre_idx8[train_pos]
    for t in pre_idx8[test_pos]:
        w = np.sum(leaf_assignments_a[train_idx] == leaf_assignments_a[t], axis=1)
        if w.sum() == 0:
            continue
        w = w / w.sum()
        for a in alphas:
            m = Ridge(alpha=a).fit(X_level8_no_const.iloc[train_idx], y8.iloc[train_idx], sample_weight=w)
            pred = m.predict(X_level8_no_const.iloc[[t]])[0]
            cv_errors_a[a].append((y8.iloc[t] - pred) ** 2)
lam_a = min(alphas, key=lambda a: np.mean(cv_errors_a[a]))
print(f"Selected ridge lambda (S augmented with price/cost/rate history, X = levels): {lam_a}")

In [ ]:
# Same nested-CV / Clark-West battery as Sections 3-4, applied to this variant (S augmented, X = levels).
n_obs8 = len(y8)
cv_records_a = []
for fold_id, (train_pos, test_pos) in enumerate(tscv_eval.split(np.arange(n_obs8))):
    y_train = y8.iloc[train_pos]
    for t in test_pos:
        w = np.sum(leaf_assignments_a[train_pos] == leaf_assignments_a[t], axis=1).astype(float)
        if w.sum() == 0:
            w = np.ones(len(train_pos))
        w = w / w.sum()
        restricted_pred = np.average(y_train, weights=w)
        m = Ridge(alpha=lam_a).fit(X_level8_no_const.iloc[train_pos], y_train, sample_weight=w)
        gtvp_pred = m.predict(X_level8_no_const.iloc[[t]])[0]
        cv_records_a.append({'fold': fold_id, 'y_true': y8.iloc[t], 'gtvp_pred': gtvp_pred, 'restricted_pred': restricted_pred})
cv_df_a = pd.DataFrame(cv_records_a)
cv_df_a['gtvp_sq_err'] = (cv_df_a['y_true'] - cv_df_a['gtvp_pred']) ** 2
cv_df_a['restricted_sq_err'] = (cv_df_a['y_true'] - cv_df_a['restricted_pred']) ** 2

ss_total_a = ((cv_df_a['y_true'] - cv_df_a['y_true'].mean()) ** 2).sum()
r2_restricted_a = 1 - cv_df_a['restricted_sq_err'].sum() / ss_total_a
r2_gtvp_a = 1 - cv_df_a['gtvp_sq_err'].sum() / ss_total_a
t_stat_a, p_ttest_a = stats.ttest_rel(cv_df_a['gtvp_sq_err'], cv_df_a['restricted_sq_err'])

cw_terms_a = cv_df_a['restricted_sq_err'] - cv_df_a['gtvp_sq_err'] + (cv_df_a['restricted_pred'] - cv_df_a['gtvp_pred']) ** 2
cw_stat_a = cw_terms_a.mean() / (cw_terms_a.std(ddof=1) / np.sqrt(len(cw_terms_a)))
p_cw_a = 1 - stats.norm.cdf(cw_stat_a)

print(f"[S augmented, X = levels] Out-of-sample R^2 - S-only: {r2_restricted_a:.4f}, S+X: {r2_gtvp_a:.4f}, incremental: {r2_gtvp_a - r2_restricted_a:.5f}")
print(f"[S augmented, X = levels] Paired t-test on squared-error differences: p={p_ttest_a:.4f}")
print(f"[S augmented, X = levels] Clark-West one-sided p-value (H1: X improves on S-only): {p_cw_a:.4f}")
if p_cw_a < 0.05:
    print("Reject H0 at the 5% level: adding price/cost/rate history to S restores a statistically significant contribution from X.")
else:
    print("Fail to reject H0 at the 5% level: adding price/cost/rate history to S does not restore a statistically significant contribution from X.")

### 8b. Keep S momentum-only, replace X with lagged growth rates

In [ ]:
# Refit on df8's rows (not df_clean's) so this variant is directly comparable to 8a on the same sample.
rf_b = RandomForestRegressor(n_estimators=500, min_samples_leaf=15, random_state=42)
rf_b.fit(S_base8, y8)
leaf_assignments_b = rf_b.apply(S_base8)
importances_b = pd.Series(rf_b.feature_importances_, index=S_base8.columns).sort_values(ascending=False)
print("RF feature importances, S unchanged (momentum-only), refit on the Section 8 sample:")
print(importances_b)

In [ ]:
X_growth8_no_const = X_growth8.drop(columns='const')
cv_errors_b = {a: [] for a in alphas}
for train_pos, test_pos in tscv.split(pre_idx8):
    train_idx = pre_idx8[train_pos]
    for t in pre_idx8[test_pos]:
        w = np.sum(leaf_assignments_b[train_idx] == leaf_assignments_b[t], axis=1)
        if w.sum() == 0:
            continue
        w = w / w.sum()
        for a in alphas:
            m = Ridge(alpha=a).fit(X_growth8_no_const.iloc[train_idx], y8.iloc[train_idx], sample_weight=w)
            pred = m.predict(X_growth8_no_const.iloc[[t]])[0]
            cv_errors_b[a].append((y8.iloc[t] - pred) ** 2)
lam_b = min(alphas, key=lambda a: np.mean(cv_errors_b[a]))
print(f"Selected ridge lambda (S momentum-only, X = lagged growth rates): {lam_b}")

In [ ]:
# Same nested-CV / Clark-West battery, applied to this variant (S momentum-only, X = lagged growth rates).
cv_records_b = []
for fold_id, (train_pos, test_pos) in enumerate(tscv_eval.split(np.arange(n_obs8))):
    y_train = y8.iloc[train_pos]
    for t in test_pos:
        w = np.sum(leaf_assignments_b[train_pos] == leaf_assignments_b[t], axis=1).astype(float)
        if w.sum() == 0:
            w = np.ones(len(train_pos))
        w = w / w.sum()
        restricted_pred = np.average(y_train, weights=w)
        m = Ridge(alpha=lam_b).fit(X_growth8_no_const.iloc[train_pos], y_train, sample_weight=w)
        gtvp_pred = m.predict(X_growth8_no_const.iloc[[t]])[0]
        cv_records_b.append({'fold': fold_id, 'y_true': y8.iloc[t], 'gtvp_pred': gtvp_pred, 'restricted_pred': restricted_pred})
cv_df_b = pd.DataFrame(cv_records_b)
cv_df_b['gtvp_sq_err'] = (cv_df_b['y_true'] - cv_df_b['gtvp_pred']) ** 2
cv_df_b['restricted_sq_err'] = (cv_df_b['y_true'] - cv_df_b['restricted_pred']) ** 2

ss_total_b = ((cv_df_b['y_true'] - cv_df_b['y_true'].mean()) ** 2).sum()
r2_restricted_b = 1 - cv_df_b['restricted_sq_err'].sum() / ss_total_b
r2_gtvp_b = 1 - cv_df_b['gtvp_sq_err'].sum() / ss_total_b
t_stat_b, p_ttest_b = stats.ttest_rel(cv_df_b['gtvp_sq_err'], cv_df_b['restricted_sq_err'])

cw_terms_b = cv_df_b['restricted_sq_err'] - cv_df_b['gtvp_sq_err'] + (cv_df_b['restricted_pred'] - cv_df_b['gtvp_pred']) ** 2
cw_stat_b = cw_terms_b.mean() / (cw_terms_b.std(ddof=1) / np.sqrt(len(cw_terms_b)))
p_cw_b = 1 - stats.norm.cdf(cw_stat_b)

print(f"[S momentum-only, X = lagged growth] Out-of-sample R^2 - S-only: {r2_restricted_b:.4f}, S+X: {r2_gtvp_b:.4f}, incremental: {r2_gtvp_b - r2_restricted_b:.5f}")
print(f"[S momentum-only, X = lagged growth] Paired t-test on squared-error differences: p={p_ttest_b:.4f}")
print(f"[S momentum-only, X = lagged growth] Clark-West one-sided p-value (H1: X improves on S-only): {p_cw_b:.4f}")
if p_cw_b < 0.05:
    print("Reject H0 at the 5% level: replacing contemporaneous levels with lagged growth rates restores a statistically significant contribution from X.")
else:
    print("Fail to reject H0 at the 5% level: replacing contemporaneous levels with lagged growth rates does not restore a statistically significant contribution from X.")

### 8c. Summary — did either respecification restore signal in X?

In [ ]:
# r2_restricted/r2_gtvp/p_value_cw are the main specification's Section 7/4 results (S momentum-only, X = margin/rate levels); no refit.
summary8 = pd.DataFrame([
    {'spec': 'Main (S=momentum, X=margin/rate levels)', 'n': len(y), 'incremental_r2': r2_gtvp - r2_restricted, 'clark_west_p': p_value_cw},
    {'spec': '8a (S=momentum+price/cost/rate, X=levels)', 'n': len(y8), 'incremental_r2': r2_gtvp_a - r2_restricted_a, 'clark_west_p': p_cw_a},
    {'spec': '8b (S=momentum, X=lagged growth rates)', 'n': len(y8), 'incremental_r2': r2_gtvp_b - r2_restricted_b, 'clark_west_p': p_cw_b},
]).set_index('spec')
print(summary8)
print()

restored = summary8['clark_west_p'] < 0.05
if not restored.any():
    print("Neither respecification restores a statistically significant contribution from X (all Clark-West p-values >= 0.05, all incremental R^2 near zero). The null result from Section 7 holds under both an augmented regime state and a lagged-growth-rate linear block: on this data, hprice/cc/margin and rate do not have detectable predictive power for housing starts once the forest's momentum-based leaves are accounted for, regardless of how X or S is specified within this family of models.")
elif restored.all():
    print("Both respecifications restore a statistically significant contribution from X -- the original null in Section 7 was driven by the specification (contemporaneous levels, momentum-only S), not by a genuine absence of effect.")
else:
    winning_spec = restored[restored].index.tolist()
    print(f"Only {winning_spec} restores a statistically significant contribution from X; the other respecification does not. This isolates which specification choice (S content vs. X timing/transformation) was responsible for the original null.")